# Differentiable optimization

A QP as a layer in a PyTorch graph: solve in the forward pass, differentiate through the solution in the backward pass.

The gradients are checked against a closed form and against `torch.autograd.gradcheck`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import cuprox

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 120, "savefig.bbox": "tight",
    "figure.facecolor": "white", "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
})
GPU = "#76B900"
REF = "#546778"

assert cuprox.__cuda_available__, (
    "This notebook must run against a CUDA build. Install with `pip install .` "
    "from the repository root; installing python/ alone does not build the "
    "extension."
)
from cuprox import _core
print(f"cuProx {cuprox.__version__} on {_core.get_device_name()}")

import torch
from cuprox.torch import solve_qp
torch.manual_seed(0)

cuProx 0.2.0 on NVIDIA RTX A6000


## The gradient is exact

For an unconstrained QP the solution is `x* = -P⁻¹q`, so the Jacobian `∂x*/∂q` is exactly `-P⁻¹`. That gives something unambiguous to check against.

In [2]:
n = 4
M = torch.randn(n, n, dtype=torch.float64)
P = M @ M.T + 3.0 * torch.eye(n, dtype=torch.float64)
q = torch.randn(n, dtype=torch.float64, requires_grad=True)

x = solve_qp(P, q)
expected = -np.linalg.solve(P.numpy(), q.detach().numpy())
print(f"forward error   {np.abs(x.detach().numpy() - expected).max():.2e}")

jac = np.zeros((n, n))
for i in range(n):
    qi = q.clone().detach().requires_grad_(True)
    seed = torch.zeros(n, dtype=torch.float64)
    seed[i] = 1.0
    solve_qp(P, qi).backward(seed)
    jac[i] = qi.grad.detach().numpy()

jac_exact = -np.linalg.inv(P.numpy())
print(f"Jacobian error  {np.abs(jac - jac_exact).max():.2e}")

# A first-order method at its default tolerance; the printed errors above
# are the real figures. The Jacobian is far tighter than the forward
# solve because it is obtained by implicit differentiation rather than by
# differentiating the iterates.
np.testing.assert_allclose(x.detach().numpy(), expected, atol=1e-4)
np.testing.assert_allclose(jac, jac_exact, atol=1e-6)

forward error   3.10e-08


Jacobian error  5.89e-10


In [3]:
ok = torch.autograd.gradcheck(
    lambda qq: solve_qp(P, qq),
    (q.clone().detach().requires_grad_(True),),
    eps=1e-6, atol=1e-4, rtol=1e-3,
)
print(f"torch.autograd.gradcheck: {ok}")
assert ok

torch.autograd.gradcheck: True


## Learning through the solver

A minimal end-to-end check: learn the linear term `q` so that the QP's solution matches a target. The only path from the loss to `q` runs through the solver, so the curve going down is itself evidence the backward pass is right.

In [4]:
torch.manual_seed(1)
n = 6
M = torch.randn(n, n, dtype=torch.float64)
P_fixed = M @ M.T + 2.0 * torch.eye(n, dtype=torch.float64)
target = torch.randn(n, dtype=torch.float64)

q_learn = torch.zeros(n, dtype=torch.float64, requires_grad=True)
opt = torch.optim.Adam([q_learn], lr=0.2)

losses = []
for _ in range(120):
    opt.zero_grad()
    loss = ((solve_qp(P_fixed, q_learn) - target) ** 2).mean()
    loss.backward()
    opt.step()
    losses.append(loss.item())

print(f"loss  {losses[0]:.4e}  ->  {losses[-1]:.4e}")
assert losses[-1] < losses[0] / 100

loss  9.5358e-01  ->  4.1012e-06


In [5]:
achieved = solve_qp(P_fixed, q_learn).detach().numpy()
print(f"max |x* - target| = {np.abs(achieved - target.numpy()).max():.2e}")

max |x* - target| = 3.16e-03
